## Compare and summarize all model performances by manually computing evaluation metrics, analyzing trade-offs in bias-variance, and justifying model selection based on statistical significance.

In [1]:
import warnings; warnings.filterwarnings('ignore')   
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('heart.csv')                          
X, y = df.drop(columns='target').values.astype(float), df['target'].values
print(X.shape, '| positive share:', y.mean().round(2))

(303, 13) | positive share: 0.54


In [2]:
def auc(t, s):
    sp, sn = s[t == 1], s[t == 0]
    return (sp[:, None] > sn[None, :]).mean() + 0.5 * (sp[:, None] == sn[None, :]).mean()

def metrics(t, p):
    pred = p > 0.5
    tp, fp, fn = np.sum(pred & (t == 1)), np.sum(pred & (t == 0)), np.sum(~pred & (t == 1))
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    q = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(acc=np.mean(pred == t), prec=prec, rec=rec, f1=2 * prec * rec / max(prec + rec, 1e-9),
                auc=auc(t, p), logloss=-np.mean(t * np.log(q) + (1 - t) * np.log(1 - q)))

In [3]:
models = {
    'LR':           lambda: LogisticRegression(max_iter=1000),
    'NB':           lambda: GaussianNB(),
    'kNN (k=15)':   lambda: KNeighborsClassifier(15),
    'kNN (k=1)':    lambda: KNeighborsClassifier(1),
    'Tree (depth 3)':  lambda: DecisionTreeClassifier(max_depth=3, random_state=0),
    'Tree (full)':     lambda: DecisionTreeClassifier(random_state=0),
    'SVM (RBF)':    lambda: SVC(probability=True, random_state=0),
    'Random Forest':lambda: RandomForestClassifier(100, random_state=0),
}
names = list(models)

def make_folds(n, k, seed): return np.array_split(np.random.default_rng(seed).permutation(n), k)
def scale(A, B): mu, sd = A.mean(0), A.std(0); return (A - mu) / sd, (B - mu) / sd     

In [4]:
res = {n: [] for n in names}; train_acc = {n: [] for n in names}
for rep in range(4):
    folds = make_folds(len(X), 5, rep)
    for k in range(5):
        te = folds[k]; tr = np.concatenate([folds[i] for i in range(5) if i != k])
        A, B = scale(X[tr], X[te])
        for n in names:
            m = models[n]().fit(A, y[tr])
            res[n].append(metrics(y[te], m.predict_proba(B)[:, 1]))
            train_acc[n].append(np.mean(m.predict(A) == y[tr]))
print('Folds:', len(res[names[0]]), '| test fold size ~', len(te), '| train fold size ~', len(tr))

Folds: 20 | test fold size ~ 60 | train fold size ~ 243


In [5]:
rows = []
for n in names:
    g = lambda c: np.array([m[c] for m in res[n]])
    rows.append([n] + [f"{g(c).mean():.3f} +- {g(c).std():.3f}" for c in ['acc', 'prec', 'rec', 'f1', 'auc']]
                + [f"{g('logloss').mean():.3f}", f"{np.mean(train_acc[n]):.3f}"])
table = pd.DataFrame(rows, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC', 'Log loss', 'Train acc'])
table

,Model,Accuracy,Precision,Recall,F1,AUC,Log loss,Train acc
0,LR,0.820 +- 0.053,0.809 +- 0.077,0.885 +- 0.048,0.843 +- 0.046,0.901 +- 0.042,0.407,0.855
1,NB,0.822 +- 0.045,0.825 +- 0.070,0.855 +- 0.045,0.838 +- 0.046,0.896 +- 0.045,0.564,0.840
2,kNN (k=15),0.833 +- 0.073,0.806 +- 0.094,0.921 +- 0.034,0.857 +- 0.064,0.906 +- 0.046,0.419,0.846
3,kNN (k=1),0.754 +- 0.050,0.767 +- 0.077,0.789 +- 0.057,0.776 +- 0.052,0.752 +- 0.048,3.397,1.000
4,Tree (depth 3),0.760 +- 0.073,0.767 +- 0.081,0.816 +- 0.109,0.785 +- 0.070,0.822 +- 0.063,0.659,0.855
5,Tree (full),0.734 +- 0.061,0.763 +- 0.078,0.752 +- 0.087,0.753 +- 0.060,0.734 +- 0.060,3.670,1.000
6,SVM (RBF),0.821 +- 0.046,0.824 +- 0.076,0.859 +- 0.050,0.838 +- 0.045,0.901 +- 0.044,0.415,0.917
7,Random Forest,0.820 +- 0.055,0.824 +- 0.074,0.858 +- 0.063,0.838 +- 0.050,0.900 +- 0.042,0.420,1.000


In [6]:
B_RUNS, bv = 30, {n: dict(err=[], bias=[], var=[]) for n in names}
rng = np.random.default_rng(0)
folds = make_folds(len(X), 5, 0)
for k in range(5):
    te = folds[k]; tr = np.concatenate([folds[i] for i in range(5) if i != k])
    preds = {n: [] for n in names}
    for b in range(B_RUNS):
        bi = rng.choice(tr, len(tr))                                   
        A, B = scale(X[bi], X[te])
        for n in names: preds[n].append(models[n]().fit(A, y[bi]).predict(B))
    for n in names:
        P = np.array(preds[n])                                   
        main = (P.mean(0) > 0.5).astype(int)                           
        bv[n]['err'].append(np.mean(P != y[te])); bv[n]['bias'].append(np.mean(main != y[te])); bv[n]['var'].append(np.mean(P != main))
bvt = pd.DataFrame({n: {c: np.mean(v) for c, v in d.items()} for n, d in bv.items()}).T
bvt.columns = ['Expected error', 'Bias (main pred wrong)', 'Variance (disagree with main)']
bvt.round(3)

,Expected error,Bias (main pred wrong),Variance (disagree with main)
LR,0.189,0.182,0.066
NB,0.184,0.185,0.053
kNN (k=15),0.182,0.171,0.074
kNN (k=1),0.236,0.231,0.103
Tree (depth 3),0.251,0.214,0.168
Tree (full),0.275,0.218,0.188
SVM (RBF),0.193,0.185,0.073
Random Forest,0.194,0.178,0.066


In [7]:
acc = {n: np.array([m['acc'] for m in res[n]]) for n in names}
best = max(names, key=lambda n: acc[n].mean())
n_f, n_tr, n_te = len(acc[best]), len(tr), len(te)
rng = np.random.default_rng(1)
rows = []
for n in names:
    if n == best: continue
    d = acc[best] - acc[n]                                            
    flips = rng.choice([-1, 1], (20000, n_f))
    p_perm = np.mean(np.abs((flips * d).mean(1)) >= abs(d.mean()) - 1e-12)   
    var_c = d.var(ddof=1) * (1 / n_f + n_te / n_tr)                    
    t_c = d.mean() / np.sqrt(var_c) if var_c > 0 else np.inf
    rows.append((n, f"{d.mean():+.3f}", f"{p_perm:.4f}", f"{t_c:.2f}", 'haan' if abs(t_c) > 2.093 else 'nahi'))
print('Best model:', best, '| accuracy', acc[best].mean().round(3))
print('Critical |t| (df=19, 5% two-sided) = 2.093\n')
pd.DataFrame(rows, columns=['Model', 'Acc diff (best - model)', 'p (permutation)', 't (corrected)', 'Significant?'])

Best model: kNN (k=15) | accuracy 0.833
Critical |t| (df=19, 5% two-sided) = 2.093



,Model,Acc diff (best - model),p (permutation),t (corrected),Significant?
0,LR,+0.013,0.2275,0.54,nahi
1,NB,+0.011,0.3509,0.40,nahi
2,kNN (k=1),+0.079,0.0001,3.01,haan
3,Tree (depth 3),+0.073,0.0003,1.98,nahi
4,Tree (full),+0.099,0.0001,2.37,haan
5,SVM (RBF),+0.012,0.2457,0.52,nahi
6,Random Forest,+0.013,0.3016,0.45,nahi


In [8]:
sig = {r[0]: r[4] == 'haan' for r in rows}
ll = {n: np.mean([m['logloss'] for m in res[n]]) for n in names}
ok = [n for n in names if n == best or (not sig[n] and acc[best].mean() - acc[n].mean() <= 0.02)]
print('Shortlist:', ok)
print('Chuna gaya (kam se kam log loss):', min(ok, key=ll.get), '| log loss', round(min(ll[n] for n in ok), 3))

Shortlist: ['LR', 'NB', 'kNN (k=15)', 'SVM (RBF)', 'Random Forest']
Chuna gaya (kam se kam log loss): LR | log loss 0.407
